# Physics of the nanoscale transistor

**Professor Nano Labs · Electronic transport**

**Try it live in your browser:** [Nanoscale transistor lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/nanoscale-transistor/)
**Watch first:** [Physics of the nanoscale transistor](https://www.youtube.com/watch?v=WBf-SfYD0bI)
**Watch later:** [Ballistic electronic transport in graphene devices](https://www.youtube.com/watch?v=2RoP7nZv474) · [Fermi Dirac vs Bose Einstein vs Classical Maxwell Boltzmann distributions in 3 minutes!](https://www.youtube.com/watch?v=e9TQs-EDiUs) · [How electricity actually works at the nanoscale?](https://www.youtube.com/watch?v=z7dVABiYnTk)

A ballistic two-dimensional transistor of width $W$. The source is grounded, with its band edge at $E=0$ and electrons filled to $E_F$; the drain's levels are lowered by $eV_D$, so its Fermi level is $E_F'=E_F-eV_D$. The gate sets the height $U_0$ of a barrier of length $L$ in the channel. Summing the Landauer formula over the transverse momentum $k_y$ gives the formula from the video,
$$\frac{I_D}{W}=I_0\sqrt{k_BT}\int_0^\infty dE_x\,T(E_x)\left[F_{-1/2}\!\left(\frac{E_F-E_x}{k_BT}\right)-F_{-1/2}\!\left(\frac{E_F-eV_D-E_x}{k_BT}\right)\right],\qquad I_0=\frac{g_se}{h}\frac{1}{\sqrt{2\pi}}\frac{\sqrt{m}}{\hbar}.$$
We use the video's $E_F=0.2$ eV and $L=50$ nm. The video does not give the effective mass; $m=1.25\,m_e$ (the same in $x$ and $y$) reproduces its plotted currents, and $g_s=2$.

The four sections match the lab's four tabs. It needs NumPy, SciPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.legendre import leggauss
from scipy.constants import e, h, hbar, k as kB, m_e
from scipy.integrate import quad
from scipy.special import expit, gamma

EF, L, m, gs = 0.20, 50e-9, 1.25 * m_e, 2          # eV, m, kg
kT = lambda T: kB * T / e                           # eV

# Fermi-Dirac integrals F_j(x) = (1/Gamma(j+1)) Int_0^inf y^j / (1 + exp(y - x)) dy, for j = -1/2 and +1/2.
# With y = t^2 the y^(-1/2) singularity goes away; split at t = sqrt(x), where the Fermi edge is.
_t, _w = leggauss(200)
def fermi_integral(j, x):
    x = np.asarray(x, float)
    xs = np.maximum(x, 0.0)[..., None]
    total = 0.0
    for a, b in ((0.0, np.sqrt(xs)), (np.sqrt(xs), np.sqrt(xs + 60.0))):
        t = a + (b - a) * (_t + 1) / 2
        total = total + np.sum((b - a) * _w / 2 * 2 * t ** (2 * j + 1) * expit(x[..., None] - t * t), axis=-1)
    return total / gamma(j + 1)
Fm = lambda x: fermi_integral(-0.5, x)
Fp = lambda x: fermi_integral(0.5, x)

def fd_quad(j, x):
    f = lambda y: y ** j * expit(x - y)
    return quad(f, 0, max(x, 0) + 60, points=[x] if x > 0 else None, limit=200, epsabs=0, epsrel=1e-12)[0] / gamma(j + 1)

xs = np.array([-30, -5, -1, 0, 0.7, 3, 7.7, 20, 100, 580])
err = max(np.max(np.abs(Fm(xs) / [fd_quad(-0.5, x) for x in xs] - 1)), np.max(np.abs(Fp(xs) / [fd_quad(0.5, x) for x in xs] - 1)))
print(f'F_-1/2 and F_1/2 against direct quadrature: largest relative error {err:.1e}')
print('nondegenerate limit, F_-1/2(-10) / e^-10 =', round(float(Fm(-10) / np.exp(-10)), 6))
print('degenerate limit, F_-1/2(100) / (2 sqrt(100/pi)) =', round(float(Fm(100) / (2 * np.sqrt(100 / np.pi))), 6))

## 1 · The Landauer picture

Above the barrier $T=1$, and the energy integral is exact: $\int_{U_0}^\infty F_{-1/2}\!\left(\frac{\mu-E_x}{k_BT}\right)dE_x=k_BT\,F_{1/2}\!\left(\frac{\mu-U_0}{k_BT}\right)$. Below it, electrons tunnel with the WKB transmission $T(E_x)=e^{-2L\sqrt{2m(U_0-E_x)}/\hbar}$ of a square barrier. With $E_x=U_0-az^2$ and $a=\hbar^2/2mL^2$ this is just $T=e^{-2z}$, which Gauss–Legendre integrates easily.

We check the result against the video's starting point, the sum over $E_x$ and $k_y$ of $T(E_x)[f_S-f_D]$, done by brute force.

In [ ]:
I0 = gs * e / h / np.sqrt(2 * np.pi) * np.sqrt(m) / hbar
_z, _wz = leggauss(64)

def transmission(Ex, U0, L=L):
    return np.where(Ex >= U0, 1.0, np.exp(-2 * L * np.sqrt(2 * m * np.maximum(U0 - Ex, 0) * e) / hbar))

def flows(U0, VD, T=300, L=L, EF=EF):
    # source -> drain and drain -> source flows, and the part that tunnels, in A per metre of width (= uA/um)
    k = kT(T)
    pref = I0 * np.sqrt(kB * T) * e                  # dE_x in eV -> J
    a = hbar ** 2 / (2 * m * L ** 2) / e             # eV
    zmax = min(np.sqrt(max(U0, 0) / a), 25.0)
    z = zmax * (_z + 1) / 2; wz = zmax * _wz / 2
    E = U0 - a * z * z
    def one(mu):
        over = k * Fp((mu - U0) / k)
        under = np.sum(wz * 2 * a * z * np.exp(-2 * z) * Fm((mu - E) / k))
        return pref * over, pref * under
    (So, Su), (Do, Du) = one(EF), one(EF - VD)
    return dict(S=So + Su, D=Do + Du, tunnel=Su - Du)

def current(U0, VD, T=300, L=L, EF=EF):
    f = flows(U0, VD, T, L, EF)
    return f['S'] - f['D']

def spectral(Ex, U0, VD, T=300, L=L, EF=EF):
    # current per unit E_x, in uA/um per meV
    k = kT(T)
    return I0 * np.sqrt(kB * T) * e * 1e-3 * transmission(Ex, U0, L) * (Fm((EF - Ex) / k) - Fm((EF - VD - Ex) / k))

def direct(U0, VD, T=300, L=L):
    # the video's sum: (g_s e/h)(1/2pi) Int dE_x T(E_x) Int dk_y [f_S - f_D], per metre of width
    k, c = kT(T), hbar ** 2 / (2 * m * e)
    def inner(Ex):
        f = lambda q: expit((EF - Ex - c * q * q) / k) - expit((EF - VD - Ex - c * q * q) / k)
        return 2 * quad(f, 0, np.sqrt((max(EF - Ex, 0) + 60 * k) / c), limit=300, epsrel=1e-10)[0]
    g = lambda Ex: transmission(Ex, U0, L) * inner(Ex)
    a = hbar ** 2 / (2 * m * L ** 2) / e
    below = quad(g, max(U0 - 60 * a, 0), U0, limit=300, epsrel=1e-9)[0] if U0 > 0 else 0
    above = quad(g, U0, max(U0, EF) + 60 * k, limit=300, epsrel=1e-10)[0]
    return gs * e / h / (2 * np.pi) * (below + above) * e

for U0, VD, T, Lc in [(0.125, 0.05, 300, 50e-9), (0.3, 0.5, 300, 50e-9), (0.4, 0.5, 300, 2e-9)]:
    print(f'U0 = {U0} eV, V_D = {VD} V, L = {Lc * 1e9:.0f} nm:  Landauer {current(U0, VD, T, Lc):.5g}   direct sum {direct(U0, VD, T, Lc):.5g}  uA/um')

fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (U0, VD, title) in zip(axs, [(0.15, 0.1, 'on: $U_0=0.15$ eV'), (0.3, 0.3, 'off: $U_0=0.30$ eV')]):
    E = np.linspace(U0 - 0.03, 0.45, 600)
    j = spectral(E, U0, VD)
    ax.fill_betweenx(E, 0, j, color='#b0561b', alpha=0.35)
    ax.plot(j, E, color='#b0561b')
    ax.axhline(U0, color='0.4'); ax.axhline(EF, color='#d9495f', ls='--'); ax.axhline(EF - VD, color='#d9495f', ls=':')
    ax.set(title=f'{title}, $V_D={VD}$ V', xlabel='current per energy (uA/um per meV)')
axs[0].set_ylabel('$E_x$ (eV)')
plt.tight_layout(); plt.show()

## 2 · Drain bias and saturation

The flow from the source does not depend on $V_D$. Raising $V_D$ only removes the flow back from the drain, and once $E_F'$ falls a few $k_BT$ below the barrier top there is nothing left to remove: the current saturates at the source flow. The video's five barriers, with the saturation currents read off its plot by eye:

In [ ]:
VDs = np.linspace(0, 0.5, 201)
barriers = [0.100, 0.125, 0.150, 0.175, 0.200]
video = [3240, 2230, 1430, 800, 420]                 # read off the video's plot
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for U0, v in zip(barriers, video):
    I = np.array([current(U0, V) for V in VDs])
    axs[0].plot(VDs, I, label=f'$U_0={U0}$ eV')
    print(f'U0 = {U0:.3f} eV: saturation current {I[-1]:6.0f} uA/um   (video: about {v})')
axs[0].set(xlabel='$V_D$ (V)', ylabel='$I_D$ (uA/um)', title='output characteristics, 300 K'); axs[0].legend()

U0 = 0.125
f = [flows(U0, V) for V in VDs]
axs[1].plot(VDs, [q['S'] for q in f], label='source -> drain')
axs[1].plot(VDs, [q['D'] for q in f], '--', label='drain -> source')
axs[1].plot(VDs, [q['S'] - q['D'] for q in f], lw=2.5, label='$I_D$')
axs[1].axvline(EF - U0, color='0.5', ls=':')
axs[1].set(xlabel='$V_D$ (V)', ylabel='flow (uA/um)', title=f'$U_0={U0}$ eV'); axs[1].legend()
plt.tight_layout(); plt.show()

# where do the linear and saturation asymptotes cross?
k = kT(300); x = (EF - U0) / k
print(f'asymptotes cross at V_D = k_BT F_1/2/F_-1/2 = {k * Fp(x) / Fm(x):.3f} V at 300 K; (2/3)(E_F - U0) = {2 / 3 * (EF - U0):.3f} V at 0 K')

## 3 · The gate and the subthreshold slope

Above $E_F$ only the Boltzmann tail of the source gets over the barrier, so $I_D\propto e^{-(U_0-E_F)/k_BT}$. Each $\ln(10)\,k_BT$ of barrier costs a decade of current: the subthreshold slope $S=\ln(10)\,k_BT/e$ is 59.5 mV/dec at 300 K and 39.7 mV/dec at 200 K. A real gate moves the barrier by only $\alpha<1$ eV per volt, so $S=\ln(10)\,k_BT/\alpha e$.

In a short channel, electrons tunnel straight through the barrier, and the gate loses its grip.

In [ ]:
U0s = np.linspace(0.1, 0.4, 121)
def slope(U0, T=300, L=L, VD=0.5, dU=1e-3):
    # barrier rise per decade, in meV
    return 1000 * 2 * dU / np.log10(current(U0 - dU, VD, T, L) / current(U0 + dU, VD, T, L))

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for T, Lc, style in [(300, 50e-9, '-'), (200, 50e-9, '-'), (300, 2e-9, '--'), (300, 1.5e-9, ':')]:
    lab = f'{T} K' + (f', L = {Lc * 1e9:g} nm' if Lc < 50e-9 else '')
    axs[0].semilogy(U0s, [current(u, 0.5, T, Lc) for u in U0s], style, label=lab)
    axs[1].plot(U0s, [slope(u, T, Lc) for u in U0s], style, label=lab)
axs[0].axvspan(EF, 0.4, color='#f0a08c', alpha=0.15)
axs[0].set(xlabel='$U_0$ (eV)', ylabel='$I_D$ (uA/um)', title='$V_D=0.5$ V'); axs[0].legend()
axs[1].set(xlabel='$U_0$ (eV)', ylabel='S (mV/dec)', ylim=(0, 200), title='local subthreshold slope'); axs[1].legend()
plt.tight_layout(); plt.show()

for T in (300, 200):
    print(f'{T} K: slope at U0 = 0.35 eV {slope(0.35, T):.1f} mV/dec, ideal ln(10) kT/e = {1000 * np.log(10) * kT(T):.1f}')
for Lc in (50e-9, 3e-9, 2e-9, 1.5e-9, 1e-9):
    f = flows(0.4, 0.5, 300, Lc); I = f['S'] - f['D']
    print(f'L = {Lc * 1e9:4.1f} nm: off current at U0 = 0.4 eV {I:9.4g} uA/um, {100 * f["tunnel"] / I:5.1f}% tunnels, slope {slope(0.4, 300, Lc):6.1f} mV/dec')

## 4 · Near equilibrium: the conductance of a perfect channel

With no barrier and a small $V_D$, $F_{-1/2}(\eta_S)-F_{-1/2}(\eta_D)\approx\frac{eV_D}{k_BT}F'_{-1/2}$, and the integral over $E_x$ gives
$$\frac{G}{W}=I_0\sqrt{k_BT}\,e\,F_{-1/2}\!\left(\frac{E_F}{k_BT}\right)\ \xrightarrow{\ T\to0\ }\ \frac{g_se^2}{h}\,\frac{1}{\pi}\frac{\sqrt{2mE_F}}{\hbar}=\frac{2e^2}{h}\cdot\frac{k_F}{\pi}.$$
The channel has $Wk_F/\pi$ transverse modes, and each conducts $2e^2/h$, however short and clean the channel.

In [ ]:
def conductance(T, EF=EF):
    return I0 * np.sqrt(kB * T) * e * Fm(EF / kT(T))          # S per metre of width
kF = lambda EF: np.sqrt(2 * m * max(EF, 0) * e) / hbar
G0 = lambda EF: 2 * e ** 2 / h * kF(EF) / np.pi

print(f'modes per um: {kF(EF) / np.pi * 1e-6:.0f}')
print(f'2e^2/h = {2 * e ** 2 / h * 1e6:.2f} uS per mode')
print(f'G at 0 K: {G0(EF) * 1e-6 * 1e3:.2f} mS/um; at 4 K {conductance(4) * 1e-3:.2f}; at 300 K {conductance(300) * 1e-3:.2f} mS/um')
print(f'R x W = {1e6 / G0(EF):.1f} Ohm um')
# small-bias check against the full current formula
print('dI/dV_D at V_D = 0.1 mV, 300 K:', round(current(0.0, 1e-4) / 1e-4, 1), ' G =', round(float(conductance(300)), 1), 'S/m')

EFs = np.linspace(-0.1, 0.4, 201)
x = np.linspace(-6, 30, 400)
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(EFs, [conductance(300, E) * 1e-3 for E in EFs], label='300 K')
axs[0].plot(EFs, [G0(E) * 1e-3 for E in EFs], '--', label='0 K formula')
axs[0].set(xlabel='$E_F$ (eV)', ylabel='G (mS/um)'); axs[0].legend()
axs[1].semilogy(x, Fm(x), lw=2.5, label='$F_{-1/2}(x)$')
axs[1].semilogy(x, np.exp(x), '--', label='$e^x$')
axs[1].semilogy(x[x > 0], 2 * np.sqrt(x[x > 0] / np.pi), '--', label=r'$2\sqrt{x/\pi}$')
axs[1].axvline(EF / kT(300), color='0.5', ls=':')
axs[1].set(xlabel='x', ylim=(1e-3, 20)); axs[1].legend()
plt.tight_layout(); plt.show()

## Check the lab's answers

These asserts confirm the numbers in the lab's "Predict first" answers.

In [ ]:
# Q1: no drain bias, no current, however low the barrier
assert current(0.1, 0.0) == 0
# Q2: off state (U0 = 0.3 eV): 95% of the current flows within 3 kT = 78 meV of the barrier top
E = np.linspace(0.25, 0.8, 20001); j = spectral(E, 0.3, 0.3)
share = np.sum(j[(E >= 0.3) & (E <= 0.3 + 3 * kT(300))]) / np.sum(j)
assert abs(share - 0.95) < 0.005 and round(3000 * kT(300)) == 78
# Q3: U0 = 0.125 eV: 82% of saturation at V_D = 0.075 V, 99% by 0.15 V, about 2240 uA/um; asymptotes cross at about 0.06 V
Isat = flows(0.125, 0.5)['S']
assert abs(current(0.125, 0.075) / Isat - 0.82) < 0.005 and current(0.125, 0.15) / Isat > 0.985 and abs(Isat - 2240) < 10
assert abs(kT(300) * Fp(0.075 / kT(300)) / Fm(0.075 / kT(300)) - 0.06) < 0.002
# Q4: back-flow 36% of the source flow at 0.05 V, 0.2% at 0.2 V
r = lambda V: flows(0.125, V)['D'] / flows(0.125, V)['S']
assert abs(r(0.05) - 0.36) < 0.005 and abs(r(0.2) - 0.002) < 0.0005
# Q5: U0 from 0.30 to 0.36 eV divides the current by about 10 at 300 K and 32 at 200 K
ratio = lambda T: current(0.30, 0.5, T) / current(0.36, 0.5, T)
assert abs(ratio(300) - 10.1) < 0.05 and abs(ratio(200) - 32) < 0.5
# Q6: alpha = 0.8 turns 59.5 mV/dec into about 74
assert abs(slope(0.35) - 59.5) < 0.3 and abs(slope(0.35) / 0.8 - 74) < 1
# Q7: L = 1.5 nm: off current 0.23 -> 1.5 uA/um, 85% tunnelling, slope about 100 mV/dec
f = flows(0.4, 0.5, 300, 1.5e-9); I = f['S'] - f['D']
assert abs(current(0.4, 0.5) - 0.23) < 0.005 and abs(I - 1.5) < 0.05 and abs(f['tunnel'] / I - 0.85) < 0.01 and abs(slope(0.4, 300, 1.5e-9) - 100) < 5
# Q8: about 63 mS/um and 815 modes per um at 4 K; 77.5 uS per mode; 16 Ohm um; 0.8% lower at 300 K
assert abs(conductance(4) * 1e-3 - 63) < 0.5 and round(kF(EF) / np.pi * 1e-6) == 815 and round(2 * e ** 2 / h * 1e6, 1) == 77.5
assert abs(1e6 / conductance(4) - 16) < 0.3 and abs(1 - conductance(300) / conductance(4) - 0.008) < 0.0005
print('all the lab\'s answers check out')

## Things to try

1. **A real gate.** Replace $U_0$ by $U_0(V_G)=U_0^{(0)}-\alpha eV_G$ with $\alpha=C_G/(C_G+C_D+C_S)$, and let the drain pull the barrier down too (drain-induced barrier lowering). How do the output curves change?
2. **Reflection above the barrier.** Replace $T=1$ above $U_0$ by the transmission of a square step. How much does it lower the on current?
3. **Graphene.** Swap the parabolic band for $E=\hbar v_F|k|$. The modes now grow as $E_F$ rather than $\sqrt{E_F}$: redo section 4 and compare with the graphene lab.
4. **Cold source.** Cut the source's Boltzmann tail off above $E_F+0.05$ eV (as an energy filter would). What happens to the subthreshold slope?